# Feature Engineering — Customer Churn Prediction

### Steps
1. Load cleaned dataset
2. Engineer target column — `churn`
3. Engineer feature columns
4. Data leakage check
5. Final feature set overview
6. Save feature dataset

## Step 1 — Load Cleaned Dataset

In [5]:
import pandas as pd
import numpy as np

CLEAN_PATH = '../../../data/customers_clean.csv'
FEATURES_PATH = '../../../data/customers_features.csv'

df = pd.read_csv(CLEAN_PATH)
print(f'Shape: {df.shape}')
df.head()

Shape: (5504, 15)


,customer_id,age,city,customer_segment,product_category,order_count,average_order_value,total_spend,discount_percentage,days_since_last_order,website_visits,support_tickets,return_count,payment_type,customer_tenure_days
0,1,58.0,Chennai,Premium,Sports,18,12622.15,227198.70,6.98,53,174,17,2,Cash,894
1,2,20.0,Chennai,Premium,Clothing,15,25515.09,382726.35,1.33,102,184,20,13,Debit Card,949
2,3,55.0,Delhi,Premium,Books,11,35057.90,385636.90,17.01,80,56,10,1,Credit Card,808
3,4,24.0,Bangalore,Regular,Furniture,17,40452.85,687698.45,36.49,275,32,12,2,Net Banking,630
4,5,58.0,Bangalore,Inactive,Clothing,46,3942.98,181377.08,33.06,149,21,7,6,Upi,599


## Step 2 — Engineer Target Column: `churn`

**Definition:** A customer is labelled as churned (`churn = 1`) if all three disengagement signals are present:

| Signal | Column | Threshold | Reasoning |
|--------|---------|-----------|----------|
| Recency | `days_since_last_order` | > 180 days | No purchase in 6+ months |
| Engagement | `website_visits` | < median | Not browsing the platform |
| Purchase history | `order_count` | < median | Low overall commitment |


In [6]:
visits_median = df['website_visits'].median()
order_median  = df['order_count'].median()

print(f'website_visits median : {visits_median}')
print(f'order_count median    : {order_median}')

website_visits median : 102.0
order_count median    : 26.0


In [7]:
df['churn'] = (
    (df['days_since_last_order'] > 180) &
    (df['website_visits'] < visits_median) &
    (df['order_count'] < order_median)
).astype(int)

churn_counts = df['churn'].value_counts()
churn_pct    = df['churn'].value_counts(normalize=True) * 100

print('Churn distribution:')
print(f'  Not churned (0): {churn_counts[0]}  ({churn_pct[0]:.1f}%)')
print(f'  Churned     (1): {churn_counts[1]}  ({churn_pct[1]:.1f}%)')

Churn distribution:
  Not churned (0): 4811  (87.4%)
  Churned     (1): 693  (12.6%)


## Step 3 — Engineer Feature Columns


### Feature 1 — `return_rate`

**Calculation:** `return_count / order_count`

**Source columns:** `return_count`, `order_count`

In [8]:
df['return_rate'] = np.where(
    df['order_count'] > 0,
    df['return_count'] / df['order_count'],
    0.0
)

print('return_rate — sample stats:')
print(df['return_rate'].describe().round(4))

return_rate — sample stats:
count    5504.0000
mean        0.4964
std         0.3159
min         0.0000
25%         0.2200
50%         0.5000
75%         0.7692
max         1.0000
Name: return_rate, dtype: float64


### Feature 2 — `purchase_frequency`

**Calculation:** `order_count / customer_tenure_days`

**Source columns:** `order_count`, `customer_tenure_days`


In [9]:
df['purchase_frequency'] = np.where(
    df['customer_tenure_days'] > 0,
    df['order_count'] / df['customer_tenure_days'],
    0.0
)

print('purchase_frequency — sample stats:')
print(df['purchase_frequency'].describe().round(6))

purchase_frequency — sample stats:
count    5504.000000
mean        0.057201
std         0.104037
min         0.000559
25%         0.014532
50%         0.027948
75%         0.053635
max         1.281250
Name: purchase_frequency, dtype: float64


### Feature 3 — `engagement_score`

**Calculation:** `website_visits / days_since_last_order`

In [10]:
df['engagement_score'] = np.where(
    df['days_since_last_order'] > 0,
    df['website_visits'] / df['days_since_last_order'],
    0.0
)

print('engagement_score — sample stats:')
print(df['engagement_score'].describe().round(6))

engagement_score — sample stats:
count    5504.000000
mean        1.841465
std         7.030496
min         0.002915
25%         0.284263
50%         0.553867
75%         1.114213
max       184.000000
Name: engagement_score, dtype: float64


### Feature 4 — `support_ticket_rate`

**Calculation:** `support_tickets / order_count`

**Source columns:** `support_tickets`, `order_count`


In [11]:
df['support_ticket_rate'] = np.where(
    df['order_count'] > 0,
    df['support_tickets'] / df['order_count'],
    0.0
)

print('support_ticket_rate — sample stats:')
print(df['support_ticket_rate'].describe().round(4))

support_ticket_rate — sample stats:
count    5504.0000
mean        0.9280
std         2.0154
min         0.0000
25%         0.1935
50%         0.3939
75%         0.7778
max        20.0000
Name: support_ticket_rate, dtype: float64


## Step 5 — Final Feature Set Overview

In [16]:
feature_cols = [
    'age',
    'customer_segment',
    'product_category',
    'payment_type',
    'average_order_value',
    'discount_percentage',
    'customer_tenure_days',
    # engineered features
    'return_rate',
    'purchase_frequency',
    'engagement_score',
    'support_ticket_rate',
]

# NOTE: days_since_last_order, website_visits, order_count are intentionally
# excluded — they were used to define the churn label. Including them would
# be data leakage: the model would just reconstruct the rule, not learn patterns.

target_col = 'churn'

print(f'Total features : {len(feature_cols)}')
print(f'Target column  : {target_col}')
print(f'Total rows     : {len(df)}')
print()
print('Feature columns:')
for f in feature_cols:
    print(f'  {f}')

Total features : 11
Target column  : churn
Total rows     : 5504

Feature columns:
  age
  customer_segment
  product_category
  payment_type
  average_order_value
  discount_percentage
  customer_tenure_days
  return_rate
  purchase_frequency
  engagement_score
  support_ticket_rate


In [17]:
# Correlation of engineered features with churn target
numeric_features = [
    'age', 'order_count', 'average_order_value', 'discount_percentage',
    'days_since_last_order', 'customer_tenure_days',
    'return_rate', 'purchase_frequency', 'engagement_score', 'support_ticket_rate'
]

correlations = df[numeric_features + ['churn']].corr()['churn'].drop('churn').sort_values()

print('Correlation with churn (sorted):')
print(correlations.round(4).to_string())

Correlation with churn (sorted):
order_count             -0.3389
purchase_frequency      -0.1075
engagement_score        -0.0891
customer_tenure_days    -0.0058
discount_percentage      0.0035
age                      0.0036
average_order_value      0.0106
return_rate              0.0185
support_ticket_rate      0.1326
days_since_last_order    0.3295


## Step 6 — Save Feature Dataset

In [19]:
output_cols = feature_cols + [target_col]
df_features = df[['customer_id'] + output_cols].copy()

df_features.to_csv(FEATURES_PATH, index=False)
print(f'Feature dataset saved to: {FEATURES_PATH}')
print(f'Shape: {df_features.shape}')
df_features.head()

Feature dataset saved to: ../../../data/customers_features.csv
Shape: (5504, 13)


,customer_id,age,customer_segment,product_category,payment_type,average_order_value,discount_percentage,customer_tenure_days,return_rate,purchase_frequency,engagement_score,support_ticket_rate,churn
0,1,58.0,Premium,Sports,Cash,12622.15,6.98,894,0.111111,0.020134,3.283019,0.944444,0
1,2,20.0,Premium,Clothing,Debit Card,25515.09,1.33,949,0.866667,0.015806,1.803922,1.333333,0
2,3,55.0,Premium,Books,Credit Card,35057.90,17.01,808,0.090909,0.013614,0.700000,0.909091,0
3,4,24.0,Regular,Furniture,Net Banking,40452.85,36.49,630,0.117647,0.026984,0.116364,0.705882,1
4,5,58.0,Inactive,Clothing,Upi,3942.98,33.06,599,0.130435,0.076795,0.140940,0.152174,0
